In [1]:
!curl -O https://www.amazontrust.com/repository/AmazonRootCA1.pem
!curl -O https://www.amazontrust.com/repository/AmazonRootCA2.pem
!curl -O https://www.amazontrust.com/repository/AmazonRootCA3.pem
!curl -O https://www.amazontrust.com/repository/AmazonRootCA4.pem
!curl -O https://certs.secureserver.net/repository/sf-class2-root.crt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1188  100  1188    0     0  17620      0 --:--:-- --:--:-- --:--:-- 18000
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1883  100  1883    0     0  31243      0 --:--:-- --:--:-- --:--:-- 31383
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   656  100   656    0     0  10551      0 --:--:-- --:--:-- --:--:-- 10580
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   737  100   737    0     0  10216      0 --:--:-- --:--:-- --:--:-- 10236
  % Total    % Received % Xferd  Average Speed   Tim

In [2]:
!cat AmazonRootCA1.pem AmazonRootCA2.pem AmazonRootCA3.pem AmazonRootCA4.pem sf-class2-root.crt > keyspaces-bundle.pem

In [3]:
!ls

 AmazonRootCA1.pem   AmazonRootCA4.pem		 sf-class2-root.crt
 AmazonRootCA2.pem  'KeySpaces Exercise.ipynb'
 AmazonRootCA3.pem   keyspaces-bundle.pem


In [ ]:
import pandas as pd

access_key = pd.read_csv("de300-keyspaces_accessKeys.csv")

access_key.head()

In [31]:
from cassandra.cluster import Cluster
from ssl import SSLContext, PROTOCOL_TLSv1_2, CERT_REQUIRED
import boto3
from cassandra_sigv4.auth import SigV4AuthProvider
import pandas as pd
import numpy as np
from cassandra import ConsistencyLevel

session.default_consistency_level = ConsistencyLevel.LOCAL_QUORUM

/tmp/ipykernel_3655/3089903178.py:9: DeprecationWarning: Setting the consistency level at the session level will be removed in 4.0. Consider using execution profiles and setting the desired consistency level to the EXEC_PROFILE_DEFAULT profile.
  session.default_consistency_level = ConsistencyLevel.LOCAL_QUORUM


In [7]:
ssl_context = SSLContext(PROTOCOL_TLSv1_2)

ssl_context.load_verify_locations('keyspaces-bundle.pem')

ssl_context.verify_mode = CERT_REQUIRED

/tmp/ipykernel_3655/3256884763.py:1: DeprecationWarning: ssl.PROTOCOL_TLSv1_2 is deprecated
  ssl_context = SSLContext(PROTOCOL_TLSv1_2)


In [8]:
boto_session = boto3.Session(
    aws_access_key_id=access_key['Access key ID'].values[0],
    aws_secret_access_key=access_key['Secret access key'].values[0],
    region_name="us-east-1"
)

In [9]:
auth_provider = SigV4AuthProvider(boto_session)

In [10]:
cluster = Cluster(
    ['cassandra.us-east-1.amazonaws.com'],
    ssl_context=ssl_context,
    auth_provider=auth_provider,
    port=9142
)

In [11]:
session = cluster.connect()

In [12]:
r = session.execute('SELECT * FROM system_schema.keyspaces')

print(r.current_rows)

[Row(keyspace_name='system_schema', durable_writes=True, replication=OrderedMapSerializedKey([('class', 'org.apache.cassandra.locator.SimpleStrategy'), ('replication_factor', '3')])), Row(keyspace_name='system_schema_mcs', durable_writes=True, replication=OrderedMapSerializedKey([('class', 'org.apache.cassandra.locator.SimpleStrategy'), ('replication_factor', '3')])), Row(keyspace_name='system', durable_writes=True, replication=OrderedMapSerializedKey([('class', 'org.apache.cassandra.locator.SimpleStrategy'), ('replication_factor', '3')])), Row(keyspace_name='system_multiregion_info', durable_writes=True, replication=OrderedMapSerializedKey([('class', 'org.apache.cassandra.locator.SimpleStrategy'), ('replication_factor', '3')])), Row(keyspace_name='de300_acharya', durable_writes=True, replication=OrderedMapSerializedKey([('class', 'org.apache.cassandra.locator.SimpleStrategy'), ('replication_factor', '3')])), Row(keyspace_name='de300_barnett', durable_writes=True, replication=OrderedMa

In [ ]:
# Starting exercise now (setup for KeySpaces completed)

In [16]:
# Downloading files
# Download needed files
!mkdir -p data

!wget -N -P data https://physionet.org/files/mimiciii-demo/1.4/ADMISSIONS.csv
!wget -N -P data https://physionet.org/files/mimiciii-demo/1.4/PRESCRIPTIONS.csv

--2026-05-28 14:57:52--  https://physionet.org/files/mimiciii-demo/1.4/ADMISSIONS.csv
Resolving physionet.org (physionet.org)... 18.13.52.205
Connecting to physionet.org (physionet.org)|18.13.52.205|:443... connected.
HTTP request sent, awaiting response... 304 Not Modified
File ‘data/ADMISSIONS.csv’ not modified on server. Omitting download.

--2026-05-28 14:57:53--  https://physionet.org/files/mimiciii-demo/1.4/PRESCRIPTIONS.csv
Resolving physionet.org (physionet.org)... 18.13.52.205
Connecting to physionet.org (physionet.org)|18.13.52.205|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1667845 (1.6M) [text/plain]
Saving to: ‘data/PRESCRIPTIONS.csv’

PRESCRIPTIONS.csv   100%[===================>]   1.59M   212KB/s    in 7.3s    

2026-05-28 14:58:00 (223 KB/s) - ‘data/PRESCRIPTIONS.csv’ saved [1667845/1667845]



In [17]:
# Loading in the datasets
admissions = pd.read_csv("data/ADMISSIONS.csv")
prescriptions = pd.read_csv("data/PRESCRIPTIONS.csv")

In [20]:
prescriptions["dose_val_rx_num"] = pd.to_numeric(
    prescriptions["dose_val_rx"],
    errors="coerce"
)

In [21]:
drug_ethnicity = prescriptions.merge(
    admissions[["subject_id", "hadm_id", "ethnicity"]],
    on=["subject_id", "hadm_id"],
    how="left"
)

drug_ethnicity = drug_ethnicity.dropna(
    subset=["ethnicity", "drug", "dose_val_rx_num"]
)

drug_summary = (
    drug_ethnicity
    .groupby(["ethnicity", "drug"], as_index=False)
    .agg(total_amount=("dose_val_rx_num", "sum"))
)

In [23]:
drug_summary["rank"] = (
    drug_summary
    .sort_values(["ethnicity", "total_amount"], ascending=[True, False])
    .groupby("ethnicity")
    .cumcount() + 1
)

top_drug_by_ethnicity = (
    drug_summary
    .query("rank == 1")
    .sort_values("ethnicity")
)

top_drug_by_ethnicity.head()

,ethnicity,drug,total_amount,rank
2,AMERICAN INDIAN/ALASKA NATIVE FEDERALLY RECOGN...,5% Dextrose,16900.0,1
93,ASIAN,Heparin,15000.0,1
195,BLACK/AFRICAN AMERICAN,Heparin Sodium,150000.0,1
287,HISPANIC OR LATINO,5% Dextrose,19950.0,1
370,HISPANIC/LATINO - PUERTO RICAN,0.9% Sodium Chloride,43663.0,1


In [24]:
session.execute("""
CREATE KEYSPACE IF NOT EXISTS yash_keyspace_exercise
WITH replication = {'class': 'SingleRegionStrategy'};
""")

In [27]:
session.execute("""
CREATE TABLE IF NOT EXISTS yash_keyspace_exercise.top_drug_by_ethnicity (
    ethnicity text,
    rank int,
    drug text,
    total_amount double,
    PRIMARY KEY ((ethnicity), rank)
) WITH CLUSTERING ORDER BY (rank ASC);
""")

In [28]:
r = session.execute("""
SELECT table_name
FROM system_schema.tables
WHERE keyspace_name = 'yash_keyspace_exercise';
""")

pd.DataFrame(r.current_rows)

,table_name
0,top_drug_by_ethnicity


In [32]:
insert_query = session.prepare("""
INSERT INTO yash_keyspace_exercise.top_drug_by_ethnicity
(ethnicity, rank, drug, total_amount)
VALUES (?, ?, ?, ?);
""")

for _, row in top_drug_by_ethnicity.iterrows():
    session.execute(
        insert_query,
        (
            str(row["ethnicity"]),
            int(row["rank"]),
            str(row["drug"]),
            float(row["total_amount"])
        )
    )

In [33]:
r = session.execute("""
SELECT *
FROM yash_keyspace_exercise.top_drug_by_ethnicity;
""")

result_df = pd.DataFrame(r.current_rows)

result_df

,ethnicity,rank,drug,total_amount
0,OTHER,1,Esmolol,5000.0
1,BLACK/AFRICAN AMERICAN,1,Heparin Sodium,150000.0
2,WHITE,1,Heparin,427700.0
3,ASIAN,1,Heparin,15000.0
4,HISPANIC/LATINO - PUERTO RICAN,1,0.9% Sodium Chloride,43663.0
5,UNKNOWN/NOT SPECIFIED,1,Epoetin Alfa,40000.0
6,UNABLE TO OBTAIN,1,0.9% Sodium Chloride,14800.0
7,AMERICAN INDIAN/ALASKA NATIVE FEDERALLY RECOGN...,1,5% Dextrose,16900.0
8,HISPANIC OR LATINO,1,5% Dextrose,19950.0


In [34]:
session.shutdown()